------------------------------------
**1. Load the food security dataset**
------------------------------------

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

DATA_PATH = "../data/finalised_dataset/food_security_fully_coded_v2.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (2886, 17)


,participant_id,Gender_Code,Work_Schedule_Code,Age_range_Code,Household_type_code,Life_satisfaction,Isolation_score,Social_support_score,food_security_score,Housing_tenure_group_code,Income_Code,Lad_code_code,msoa_code_code,imd_decile,green_space_pct,park_distance_m,crime_rate_per_1000
0,01g1puj0zbtyyl68d9l01g1pujrxajj5,1,2,3,1,4,0,2,9,1,4,9000013,2000387,7,66.5,411.026450,113.597404
1,01h0q6h87uvwo01h0qeji2vxnl4z78ik,1,2,6,1,6,1,1,0,2,4,9000015,2006882,7,95.2,722.929373,61.229398
2,01nhwhotteqvsg5qfl301nhwho84uknz,2,7,4,2,8,2,2,9,3,0,9000033,2000981,7,89.8,438.816681,372.933205
3,020yzynec712xmojk020yzynd1hjo9pl,1,2,9,2,9,0,2,0,1,3,9000009,2000276,5,90.2,360.395918,81.460788
4,029tb4078h9gvgwasyk029tb49nw0foj,2,2,9,3,7,2,2,0,2,3,9000029,2000852,10,94.3,454.377365,64.105795


In [6]:
print(
    df["food_security_score"]
    .value_counts()
    .sort_index()
)

print(
    df["food_security_score"]
    .describe()
)

assert df["food_security_score"].between(0, 9).all()

food_security_score
0    1473
1     233
2     158
3     149
4     115
5     109
6     125
7     144
8     187
9     193
Name: count, dtype: int64
count    2886.000000
mean        2.422730
std         3.169846
min         0.000000
25%         0.000000
50%         0.000000
75%         5.000000
max         9.000000
Name: food_security_score, dtype: float64


----------------------------------------
**2. Create the food-security classes**
---------------------------------------

In [8]:
def create_food_security_class(score):
    if score <= 1:
        return 0 # High
    elif score <= 3:
        return 1 # Marginal
    elif score <= 6:
        return 2 # Low
    else:
        return 3 # Very Low

df["target"] = (
    df["food_security_score"]
    .apply(create_food_security_class)
)

class_names = {
    0: "High",
    1: "Marginal",
    2: "Low",
    3: "Very Low"
}

print(
    df["target"]
    .map(class_names)
    .value_counts()
)

target
High        1706
Very Low     524
Low          349
Marginal     307
Name: count, dtype: int64


------------------------------------
**3. Build X and y**
------------------------------------

In [9]:
y = df["target"].copy()

X = df.drop(
    columns=[
        "participant_id",
        "food_security_score",
        "target"
    ]
).copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print(X.columns.tolist())

X shape: (2886, 15)
y shape: (2886,)
['Gender_Code', 'Work_Schedule_Code', 'Age_range_Code', 'Household_type_code', 'Life_satisfaction', 'Isolation_score', 'Social_support_score', 'Housing_tenure_group_code', 'Income_Code', 'Lad_code_code', 'msoa_code_code', 'imd_decile', 'green_space_pct', 'park_distance_m', 'crime_rate_per_1000']


In [10]:
geographic_ids = [
    "Lad_code_code",
    "msoa_code_code"
]

X = X.drop(columns=geographic_ids)

**4. Split data into Train and Test**
-------------------------------------

In [11]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

In [12]:
print(f"Train:      {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test.shape}")

Train:      (2020, 13)
Validation: (433, 13)
Test:       (433, 13)


In [13]:
def show_distribution(name, y):
    counts = y.value_counts().sort_index()
    percentages = (
        y.value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )

    result = pd.DataFrame({
        "count": counts,
        "percentage": percentages
    })

    result.index = result.index.map(class_names)

    print(f"\n{name}")
    print(result)


show_distribution("TRAIN", y_train)
show_distribution("VALIDATION", y_val)
show_distribution("TEST", y_test)


TRAIN
          count  percentage
target                     
High       1194       59.11
Marginal    215       10.64
Low         244       12.08
Very Low    367       18.17

VALIDATION
          count  percentage
target                     
High        256       59.12
Marginal     46       10.62
Low          52       12.01
Very Low     79       18.24

TEST
          count  percentage
target                     
High        256       59.12
Marginal     46       10.62
Low          53       12.24
Very Low     78       18.01


**Verify no Overlap**
----------------------

In [15]:
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)

assert len(X_train) + len(X_val) + len(X_test) == len(X)

print("Split integrity checks passed.")

Split integrity checks passed.


In [16]:
from pathlib import Path

split_dir = Path("../data/finalised_dataset/splits/food_security")
split_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(split_dir / "X_train.csv", index=False)
X_val.to_csv(split_dir / "X_validation.csv", index=False)
X_test.to_csv(split_dir / "X_test.csv", index=False)

y_train.to_csv(split_dir / "y_train.csv", index=False)
y_val.to_csv(split_dir / "y_validation.csv", index=False)
y_test.to_csv(split_dir / "y_test.csv", index=False)

In [17]:
for col in X.columns:
    print(f"\n{col}")
    print("dtype:", X[col].dtype)
    print("unique:", X[col].nunique())
    print("missing:", X[col].isna().sum())
    print("values:", sorted(X[col].dropna().unique())[:20])


Gender_Code
dtype: int64
unique: 4
missing: 0
values: [0, 1, 2, 3]

Work_Schedule_Code
dtype: int64
unique: 14
missing: 0
values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]

Age_range_Code
dtype: int64
unique: 14
missing: 0
values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

Household_type_code
dtype: int64
unique: 6
missing: 0
values: [1, 2, 3, 4, 5, 6]

Life_satisfaction
dtype: int64
unique: 10
missing: 0
values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Isolation_score
dtype: int64
unique: 3
missing: 0
values: [0, 1, 2]

Social_support_score
dtype: int64
unique: 3
missing: 0
values: [0, 1, 2]

Housing_tenure_group_code
dtype: int64
unique: 9
missing: 0
values: [0, 1, 2, 3, 4, 5, 6, 7, 8]

Income_Code
dtype: int64
unique: 6
missing: 0
values: [0, 1, 2, 3, 4, 5]

imd_decile
dtype: int64
unique: 10
missing: 0
values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

green_space_pct
dtype: float64
unique: 371
missing: 0
values: [7.2, 15.9, 19.8, 20.8, 21.8, 23.1, 27.0, 27.1, 28.8, 29.7, 29.9, 30.6, 32.

In [18]:
X_train.describe().T

,count,mean,std,min,25%,50%,75%,max
Gender_Code,2020.0,1.243069,0.488426,0.000000,1.000000,1.000000,1.000000,3.000000
Work_Schedule_Code,2020.0,3.740099,2.390189,0.000000,2.000000,3.000000,5.000000,13.000000
Age_range_Code,2020.0,5.382673,2.411915,1.000000,4.000000,5.000000,7.000000,14.000000
Household_type_code,2020.0,3.382178,1.673876,1.000000,2.000000,4.000000,5.000000,6.000000
Life_satisfaction,2020.0,6.936634,1.880255,1.000000,6.000000,7.000000,8.000000,10.000000
Isolation_score,2020.0,1.299010,0.856001,0.000000,0.000000,2.000000,2.000000,2.000000
Social_support_score,2020.0,1.656931,0.545708,0.000000,1.000000,2.000000,2.000000,2.000000
Housing_tenure_group_code,2020.0,3.240594,1.704388,0.000000,2.000000,3.000000,5.000000,8.000000
Income_Code,2020.0,2.546535,1.813305,0.000000,1.000000,3.000000,4.000000,5.000000
imd_decile,2020.0,4.895545,2.347367,1.000000,3.000000,4.000000,7.000000,10.000000


**Define Feature Groups**
-------------------------

In [ ]:
nominal_features = [
    "Gender_Code",
    "Work_Schedule_Code",
    "Household_type_code",
    "Housing_tenure_group_code"
]

ordinal_features = [
    "Age_range_Code",
    "Income_Code",
    "imd_decile"
]

continuous_features = [
    "Life_satisfaction",
    "Isolation_score",
    "Social_support_score",
    "green_space_pct",
    "park_distance_m",
    "crime_rate_per_1000"
]